# LACT_sim ROOT -> pylast quicklook

这个 notebook 放在 LACT_sim 仓库里，用于检查从 LACT_sim 输出 ROOT 到 pylast 读取和画图的完整链路。

推荐流程是：

1. 用 `run_corsika_trace` 生成 `lact_event_root_v1` ROOT 文件。
2. 用独立的 `lact-pylast` 环境读取 ROOT。
3. 画两个标准检查图：阵列芯位/望远镜分布图，以及触发望远镜相机图。

如果你已经生成好了 ROOT 文件，可以从第 3 节开始运行。


## 1. 编译 LACT_sim

在服务器上，如果 `root-config --version` 能输出版本，就说明 ROOT 已在当前 shell 可用。普通更新后可以直接用项目原来的 `make`：


In [ ]:
%%bash
set -euo pipefail

make hessio
make run_corsika_trace -j"${SLURM_CPUS_PER_TASK:-8}"

strings ./build/run_corsika_trace | grep "opening streaming lact_event ROOT writer" || true


## 2. 运行一个 CORSIKA/EventIO 文件生成 ROOT

默认使用 ROOT-only full-response 配置，避免同时写 HDF5 占用空间。当前配置默认 `source.max_shower_events=-1`，也就是跑完整输入文件。快速测试时可以临时把配置改成正数。


In [ ]:
from pathlib import Path

# 改成你的输入 CORSIKA/EventIO 路径。
INPUT_EVENTIO = Path("/eos/lhaaso/simulation/lactmc/prod1/point_gamma/corsika/zenith_20/azimuth_0/100GeV_1000GeV/run000001/lact_prod1_corsika_particle_gamma_energy_100.0_1000.0_zenith_20.0_azimuth_0.0_run_1_event_0.zst")

# ROOT-only: only writes run_logs/lact_root_only_full_response/lact_events.root
CFG = Path("configs/examples/corsika_lact_root_only_full_response.cfg")
ROOT_FILE = Path("run_logs/lact_root_only_full_response/lact_events.root")

# If you intentionally used HDF5 + ROOT instead, switch to these paths:
# CFG = Path("configs/examples/corsika_lact_root_full_response.cfg")
# ROOT_FILE = Path("run_logs/lact_root_full_response/lact_events.root")

RUN_LOG = ROOT_FILE.parent / "run.log"

print("config:", CFG)
print("input:", INPUT_EVENTIO)
print("ROOT output:", ROOT_FILE)
print("run log:", RUN_LOG)


In [ ]:
import subprocess
import sys

ROOT_FILE.parent.mkdir(parents=True, exist_ok=True)
cmd = ["./build/run_corsika_trace", str(CFG), str(INPUT_EVENTIO)]
print(" ".join(cmd))
print("logging to:", RUN_LOG)

with RUN_LOG.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        log.write(line)
    return_code = proc.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

print("run log saved:", RUN_LOG)


## 3. 用 pylast 读取 ROOT

下面的 cell 需要 Jupyter kernel 使用独立的 `lact-pylast` 环境。检查方式：`sys.executable` 应该指向 `/home/lhaaso/huangyiyun/conda/envs/lact-pylast/bin/python` 一类路径。


In [ ]:
from pathlib import Path
import os
import sys

# Keep matplotlib/cache writes away from AFS/home quota.
os.environ.setdefault("MPLCONFIGDIR", "/home/lhaaso/huangyiyun/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

if not ROOT_FILE.exists():
    fallback = Path("run_logs/lact_root_full_response/lact_events.root")
    if fallback.exists():
        ROOT_FILE = fallback

print("python:", sys.executable)
print("ROOT_FILE:", ROOT_FILE)
print("exists:", ROOT_FILE.exists())


In [ ]:
from pylast.io import LactEventSource
from pylast.visualize import plot_event_cores, plot_event_cameras, plot_lact_root_quicklook

EVENT_INDEX = 0
MAX_EVENTS = 10
IMAGE_LEVEL = "dl0"

source = LactEventSource(str(ROOT_FILE), max_events=MAX_EVENTS)
event = source[EVENT_INDEX]
triggered_tels = list(getattr(event.simulation, "triggered_tels", []))

print("event_id:", event.event_id)
print("run_id:", event.run_id)
print("loaded max_events:", getattr(source, "max_events", None))
print("subarray telescope ids:", sorted(source.subarray.tels.keys()))
print("triggered telescope ids:", triggered_tels)
print("DL0 telescope ids:", sorted(event.dl0.tels.keys()))
print("R1 telescope ids:", sorted(event.r1.tels.keys()))


In [ ]:
# Inspect one telescope payload. Prefer a triggered telescope when available.
tel_id = triggered_tels[0] if triggered_tels else sorted(event.dl0.tels.keys())[0]
dl0_camera = event.dl0.tels[tel_id]
r1_camera = event.r1.tels[tel_id]

print("tel_id:", tel_id)
print("DL0 image shape:", dl0_camera.image.shape)
print("DL0 peak_time shape:", dl0_camera.peak_time.shape)
print("R1 waveform shape:", r1_camera.waveform.shape)
print("R1 gain_selection shape:", r1_camera.gain_selection.shape)
print("DL0 total p.e.:", float(dl0_camera.image.sum()))
print("R1 waveform total p.e.:", float(r1_camera.waveform.sum()))


## 4. 图 1：芯位/望远镜分布图

这个 cell 画阵列坐标中的望远镜分布、shower core、event 方向、telescope pointing 方向，并用红圈标记真正触发的望远镜。


In [ ]:
array_result = plot_event_cores(
    root_file=ROOT_FILE,
    event_index=EVENT_INDEX,
    max_events=MAX_EVENTS,
    image_level=IMAGE_LEVEL,
    show_sdp_planes=True,
)

array_result["figure"]


## 5. 图 2：触发望远镜相机图

这个 cell 只画触发望远镜的相机图。若要检查完整 ROOT 里非触发但有 p.e. 的望远镜，可以把 `include_non_triggered=True`。


In [ ]:
camera_result = plot_event_cameras(
    root_file=ROOT_FILE,
    event_index=EVENT_INDEX,
    max_events=MAX_EVENTS,
    image_level=IMAGE_LEVEL,
    include_non_triggered=False,
)

camera_result["figure"]


## 6. 可选：保存全部 quicklook PNG


In [ ]:
OUTPUT_DIR = ROOT_FILE.parent / "pylast_visualize"

save_result = plot_lact_root_quicklook(
    root_file=ROOT_FILE,
    output_dir=OUTPUT_DIR,
    event_index=EVENT_INDEX,
    max_events=MAX_EVENTS,
    image_level=IMAGE_LEVEL,
    show=False,
)

for name, path in save_result["paths"].items():
    print(f"{name}: {path}")
